# Solutions 00: Distributions, Divergences, and Numerics

Solutions to the four exercises of **Lab 00** (`lab-00-distributions-divergences-numerics`).
All four solutions are **live**: every cell executed during the build, nothing is gated behind
a flag, and every solution ends in an `assert` or a printed check that the interpretation cell
then reads back. Attempt the exercises yourself before opening this file; the point of a
Tier 1 lab is the fifteen minutes you spend being wrong before the assertion tells you why.

In [1]:
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")   # widget progress bars crash some notebook stacks; plain logs are fine

import sys, math
sys.path.insert(0, "../code")

import torch
import torch.nn.functional as F

torch.manual_seed(0)
print(f"torch {torch.__version__}")

def rand_dist(*shape, peak=1.0, generator=None):
    '''Random categorical distribution via softmax of scaled Gaussian logits.'''
    return F.softmax(peak * torch.randn(*shape, generator=generator), dim=-1)

torch 2.13.0+cpu


## Exercise 1: break bf16 on purpose

**The exercise.** Rerun the lab's §7 precision experiment (compute a KL in fp64 as the
reference, then repeat it with the `log_softmax` performed in fp32 and in bf16) with the logit
scale at 1, 12, and 40. At which scale does bf16 loss math first disagree with fp32 in the
second significant digit? Relate the answer to the mantissa widths: bf16 keeps roughly 2 to 3
significant decimal digits, so the disagreement should appear once the computation demands
more than that.

**The approach.** The question is really asking: how does bf16's fixed *relative* precision
interact with a computation whose *magnitude* grows? bf16 has a 7-bit mantissa, so every
stored value carries a relative error of up to about `7.8e-3 / 2`, roughly 0.4%, regardless
of how large the value is. Scaling the logits up makes each individual logit's *absolute*
rounding error larger (a logit near 120 is stored on a grid with spacing near 1.0), but it
also makes the KL itself larger, so the two effects partly cancel in the relative error of
the final number. The plan: reuse the lab's exact `kl_in` computation, sweep a finer set of
scales than the exercise names (1, 2, 4, 8, 12, 20, 40) so the shape of the error curve is
visible, and locate the first scale where the bf16 result disagrees with the fp64 reference
by more than one part in a thousand (a corrupted third significant digit) and by more than
one part in a hundred (a corrupted second significant digit). The second threshold is what
the exercise asks about, and whether it is ever crossed is an empirical question the cell
answers rather than assumes.

In [2]:
g = torch.Generator().manual_seed(5)
base_p = torch.randn(64, 256, generator=g)     # same shape and seed family as lab 00 section 7
base_q = torch.randn(64, 256, generator=g)

def kl_at(scale, dtype):
    zp, zq = scale * base_p, scale * base_q
    lp = F.log_softmax(zp.to(dtype), -1).float()
    lq = F.log_softmax(zq.to(dtype), -1).float()
    return float((lp.exp() * (lp - lq)).sum(-1).mean())

scales = (1, 2, 4, 8, 12, 20, 40)
err32, err16 = {}, {}
print(f"{'scale':>6} {'fp64 ref KL':>12} {'fp32 rel err':>13} {'bf16 rel err':>13}")
for s in scales:
    ref = kl_at(s, torch.float64)
    err32[s] = abs(kl_at(s, torch.float32) - ref) / ref
    err16[s] = abs(kl_at(s, torch.bfloat16) - ref) / ref
    print(f"{s:>6} {ref:>12.4f} {err32[s]:>13.2e} {err16[s]:>13.2e}")

third_digit  = [s for s in scales if err16[s] > 1e-3]   # rel err > 1e-3: 3rd digit corrupted
second_digit = [s for s in scales if err16[s] > 1e-2]   # rel err > 1e-2: 2nd digit corrupted

print(f"\nscales where bf16 corrupts the 3rd significant digit: {third_digit}")
print(f"scales where bf16 corrupts the 2nd significant digit: {second_digit}")

assert all(e < 1e-5 for e in err32.values()), "fp32 loss math stays clean at every scale"
assert len(third_digit) > 0, "bf16 must corrupt the third digit somewhere in the sweep"
assert max(err16.values()) < 1e-2, "on this tensor the second digit is never corrupted"
assert min(err16.values()) > 10 * max(err32.values()), "bf16 is always far noisier than fp32"
print("\nchecked: fp32 clean everywhere; bf16 error lives in the 3rd significant digit")

 scale  fp64 ref KL  fp32 rel err  bf16 rel err
     1       0.9836      6.06e-08      4.36e-04


     2       3.7309      0.00e+00      1.41e-03
     4      10.5003      9.08e-08      3.41e-04


     8      22.5800      0.00e+00      4.92e-04
    12      34.1861      1.12e-07      1.80e-03


    20      57.2573      1.33e-07      7.38e-05
    40     114.8574      6.64e-08      4.96e-04

scales where bf16 corrupts the 3rd significant digit: [2, 12]
scales where bf16 corrupts the 2nd significant digit: []

checked: fp32 clean everywhere; bf16 error lives in the 3rd significant digit


**Interpretation.** The printed table shows fp32 holding a relative error below `1e-5` at
every scale, which the first assertion confirms, while bf16 sits between about `1e-4` and
`2e-3`, peaking at scale 12. Two findings, one expected and one that corrects the exercise's
premise.

The expected one: the bf16 error is consistent with the digit budget. A relative error of
`1e-3` to `2e-3` means the third significant digit of the loss is noise, which is exactly what
"roughly 2 to 3 significant digits" predicts for a 7-bit mantissa (machine eps `7.8e-3`, and
averaging over 64 rows claws back a little accuracy, which is why the error lands below eps
rather than at it). The printed list shows the third digit goes bad at scales 2 and 12, with
no monotone trend in between: where inside the eps band the error lands depends on how the
individual rounding errors happen to cancel, which is itself worth knowing (bf16 noise is not
a smooth function of scale you can extrapolate; it wanders inside the digit budget). At scale
12 the logits span roughly -40 to +40, so the largest values sit on a bf16 grid with spacing
around 0.25, and errors of that size on the log-probabilities feed straight into a KL of only
~34 nats.

The honest correction: on this tensor the **second** significant digit is never corrupted at
any of the exercise's scales, and the third assertion pins that down (`max` bf16 relative
error below `1e-2`). The reason is visible in the table's reference column: scaling the
logits up grows the KL itself (from ~1 nat at scale 1 to ~115 nats at scale 40) at least as
fast as it grows the absolute rounding error, so the *relative* error does not keep climbing;
it actually falls again between scale 12 and scale 40. So the accurate statement is not "bf16
gets relatively worse as logits grow" but "bf16 holds the loss to about three digits no
matter what, and the third digit is already noise at realistic post-temperature logit
scales". That is still ample reason for the lab's rule (model in bf16, divergence in fp32):
a loss whose third digit is noise cannot resolve the small late-training improvements a
distillation run is trying to measure.

## Exercise 2: Pinsker in reverse

**The exercise.** Pinsker's inequality bounds TVD by KL
($\mathrm{TVD} \le \sqrt{\mathrm{KL}/2}$), but no bound exists in the other direction.
Construct a sequence of distribution pairs with TVD $\to 0$ and KL $\to \infty$, by moving a
shrinking amount of probability mass onto a token where $q$ is astronomically small.

**The approach.** The question is really asking *why* the two divergences can disagree by an
unbounded factor: TVD adds up absolute probability differences, so a tiny amount of misplaced
mass contributes a tiny amount, full stop. KL weights each token's contribution by the
log-ratio $\log(p_i/q_i)$, and a log-ratio has no ceiling: it grows without limit as $q_i$
shrinks. So the recipe is to make the *mass* small (which keeps TVD small) while making the
*ratio* enormous (which blows KL up). Concretely, on a two-token vocabulary, let $p$ put a
shrinking mass $\varepsilon$ on token 2, and let $q$ put the far smaller mass
$a = e^{-1/\varepsilon^2}$ there. Then TVD $= \varepsilon - a \approx \varepsilon \to 0$,
while the KL is dominated by the term
$\varepsilon \cdot \log(\varepsilon / a) \approx \varepsilon \cdot 1/\varepsilon^2 =
1/\varepsilon \to \infty$. One numerical care point: $e^{-1/\varepsilon^2}$ underflows any
float format almost immediately (for $\varepsilon = 0.03$ it is $e^{-1111}$), so the KL must
be computed from $\log a = -1/\varepsilon^2$ directly, in log space, exactly the discipline
the lab's §2 taught. The cell cross-checks the log-space formula against a plain torch KL at
the one $\varepsilon$ where $a$ is still representable, so the shortcut is verified rather
than trusted.

In [3]:
def pair_stats(eps):
    # p = [1-eps, eps],  q = [1-a, a]  with  log a = -1/eps^2  (a itself may underflow).
    log_a = -1.0 / eps**2
    a = math.exp(log_a)                       # 0.0 once it underflows fp64; that is fine
    tvd_v = eps - a                           # 0.5 * (|p1-q1| + |p2-q2|) = eps - a
    kl_v = (1 - eps) * (math.log(1 - eps) - math.log1p(-a)) \
         + eps * (math.log(eps) - log_a)      # log-space: never touches a itself
    return tvd_v, kl_v

# Cross-check the closed form against a plain torch KL where a is still representable.
eps0 = 0.1
a0 = math.exp(-1.0 / eps0**2)                 # e^-100 ~ 3.7e-44, representable in fp64
p0 = torch.tensor([1 - eps0, eps0], dtype=torch.float64)
q0 = torch.tensor([1 - a0, a0], dtype=torch.float64)
kl_torch = float((p0 * (p0.log() - q0.log())).sum())
tvd_ref, kl_ref = pair_stats(eps0)
assert abs(kl_torch - kl_ref) / kl_ref < 1e-12, "log-space formula must match torch's KL"

print(f"{'eps':>8} {'TVD(p,q)':>10} {'KL(p||q)':>12} {'KL/TVD':>12} {'sqrt(KL/2)':>11}")
rows = []
for eps in (0.1, 0.03, 0.01, 0.003, 0.001):
    tvd_v, kl_v = pair_stats(eps)
    rows.append((eps, tvd_v, kl_v))
    print(f"{eps:>8} {tvd_v:>10.5f} {kl_v:>12.2f} {kl_v/tvd_v:>12.1f} {math.sqrt(kl_v/2):>11.2f}")

tvds = [r[1] for r in rows]; kls = [r[2] for r in rows]
assert all(a > b for a, b in zip(tvds, tvds[1:])), "TVD must fall along the sequence"
assert all(a < b for a, b in zip(kls, kls[1:])), "KL must grow along the sequence"
assert tvds[-1] < 2e-3 and kls[-1] > 900, "endpoint: TVD ~ 1e-3 while KL ~ 1e3"
assert all(t <= math.sqrt(k / 2) for _, t, k in rows), "Pinsker itself still holds"
print("\nchecked: TVD -> 0 while KL -> inf on the same sequence; Pinsker never violated")

     eps   TVD(p,q)     KL(p||q)       KL/TVD  sqrt(KL/2)
     0.1    0.10000         9.67         96.7        2.20
    0.03    0.03000        33.20       1106.6        4.07
    0.01    0.01000        99.94       9994.4        7.07
   0.003    0.00300       333.31     111104.3       12.91
   0.001    0.00100       999.99     999992.1       22.36

checked: TVD -> 0 while KL -> inf on the same sequence; Pinsker never violated


**Interpretation.** The table is the whole argument. As $\varepsilon$ falls from 0.1 to
0.001, TVD falls right alongside it (first column tracks $\varepsilon$ almost exactly, since
$a$ is negligible), while KL *grows* from about 10 nats to about 1000, matching the
$1/\varepsilon$ prediction. The KL/TVD ratio climbs past $10^6$, and nothing stops it: run
$\varepsilon$ smaller and the ratio grows without bound. The final assertion confirms Pinsker
is never violated along the way, because Pinsker's promise runs in the other direction only
(small KL forces small TVD; the last column, $\sqrt{\mathrm{KL}/2}$, is a huge and useless
upper bound here, which is the point).

The mechanism, stated once more because it is the transferable part: TVD is a sum of
*absolute differences*, and the misplaced mass is only $\varepsilon$, so TVD cannot exceed
$\varepsilon$. KL is a sum of *mass times log-ratio*, and the log-ratio on token 2 is
$1/\varepsilon^2$, so the product $\varepsilon \cdot 1/\varepsilon^2$ diverges even as the
mass vanishes. Operationally: a measured TVD of 0.001 between student and teacher certifies
nothing about their KL, because the student may have assigned an astronomically small (or
zero) probability to a rare token the teacher still cares about. Lab 02's tail-mass
discussion is this table wearing production clothes: the tail is exactly where tiny masses
with enormous log-ratios live, and any cache or comparison that truncates the tail is blind
to unbounded KL hiding in it.

## Exercise 3: your own generator, the chi-square divergence

**The exercise.** The $\chi^2$ divergence has generator $f(t) = (t-1)^2$. Add it to the §5
f-divergence harness, verify it against a direct formula, check whether it is bounded, and
decide whether you would train against it, considering what squaring the ratio does to the
§9 variance story.

**The approach.** Three steps. First, plug $f(t) = (t-1)^2$ into the generic
$D_f(p\|q) = \sum_i q_i f(p_i/q_i)$ harness and verify it equals the direct formula
$\sum_i (p_i - q_i)^2 / q_i$, which follows by expanding
$q_i (p_i/q_i - 1)^2 = (p_i - q_i)^2/q_i$. Second, verify the identity that explains
everything else about this divergence:
$\chi^2(p\|q) = \mathbb{E}_q[r^2] - 1 = \mathrm{Var}_q[r]$ where $r = p/q$ is the importance
ratio (the correction factor that reweights samples drawn from $q$ to stand in for samples
from $p$; $\mathbb{E}_q[r] = 1$ always, so the variance is the second moment minus one).
Third, probe boundedness by shrinking the student's probability on a token the teacher is
confident about, and race $\chi^2$ against forward KL and JSD on the same sequence: KL should
grow *linearly* in the logit gap (it contains one power of $\log r$), $\chi^2$ should grow
*exponentially* (it contains $r$ itself, squared), and JSD should saturate at $\log 2$.

In [4]:
def f_div(p, q, f):
    t = p / q
    return (q * f(t)).sum(-1)

g = torch.Generator().manual_seed(3)
p = rand_dist(6, 48, generator=g)
q = rand_dist(6, 48, generator=g)

chi2_gen    = f_div(p, q, lambda t: (t - 1) ** 2)      # via the generator
chi2_direct = ((p - q) ** 2 / q).sum(-1)               # direct formula
chi2_moment = (q * (p / q) ** 2).sum(-1) - 1.0         # E_q[r^2] - 1
assert torch.allclose(chi2_gen, chi2_direct, atol=1e-6), "generator != direct formula"
assert torch.allclose(chi2_gen, chi2_moment, atol=1e-6), "chi2 must equal Var_q[r]"
print(f"chi2 via generator, direct formula, and E_q[r^2]-1 all agree "
      f"(mean {float(chi2_gen.mean()):.4f})")

# Unboundedness race: teacher confident on token 0, student mass on it shrinking.
print(f"\n{'student logit':>14} {'~log r':>7} {'fwd KL':>10} {'chi2':>14} {'JSD':>8}")
t_logits = torch.tensor([8.0, 0.0, 0.0, 0.0])
p_conf = F.softmax(t_logits, -1)
kls, chis = [], []
for s0 in (-5.0, -10.0, -20.0, -30.0):
    s_logits = torch.tensor([s0, 0.0, 0.0, 0.0])
    q_s = F.softmax(s_logits, -1)
    kl_v  = float((p_conf * (p_conf.log() - q_s.log())).sum())
    chi_v = float(((p_conf - q_s) ** 2 / q_s).sum())
    mm = 0.5 * (p_conf + q_s)
    jsd_v = float(0.5 * (p_conf * (p_conf.log() - mm.log())).sum()
                  + 0.5 * (q_s * (q_s.log() - mm.log())).sum())
    kls.append(kl_v); chis.append(chi_v)
    print(f"{s0:>14} {abs(s0):>7.0f} {kl_v:>10.2f} {chi_v:>14.4g} {jsd_v:>8.4f}")

assert chis[-1] / chis[0] > 1e5, "chi2 grows exponentially in the logit gap: unbounded, fast"
assert kls[-1] / kls[0] < 10, "forward KL grows only linearly in the same gap"
assert jsd_v < math.log(2) + 1e-6, "JSD stays under log 2 no matter what"
print("\nchecked: chi2 explodes exponentially where KL grows linearly and JSD saturates")

chi2 via generator, direct formula, and E_q[r^2]-1 all agree (mean 6.4322)

 student logit  ~log r     fwd KL           chi2      JSD
          -5.0       5       6.09          444.3   0.6812
         -10.0      10      11.08      6.595e+04   0.6891
         -20.0      20      21.07      1.453e+09   0.6892
         -30.0      30      31.06      3.199e+13   0.6892

checked: chi2 explodes exponentially where KL grows linearly and JSD saturates


**Interpretation.** The first check confirms the bookkeeping: the generator form, the
direct formula $\sum (p-q)^2/q$, and the moment form $\mathbb{E}_q[r^2] - 1$ are the same
number, so $\chi^2$ is a legitimate member of the §5 family and is *literally the variance of
the importance ratio*.

The race table answers the boundedness question emphatically. Walking the student's logit on
the teacher's favourite token from -5 to -30 (so $\log r$ on that token grows by 25), forward
KL grows from about 6 to about 31 nats, a factor near 5, because KL charges one power of
$\log r$. Meanwhile $\chi^2$ grows by more than ten orders of magnitude, because it charges
$r^2 = e^{2\log r}$: every unit of logit gap *doubles-and-then-some* the loss, twice over.
JSD, as always, sits below $\log 2 \approx 0.693$.

Would I train against it? No, and the moment form says why in one line. The gradient signal
of $\chi^2$ is dominated by whichever token currently has the largest ratio $r$, raised to
the second power, so a single rare token where the student lags the teacher badly does not
just contribute the largest term (already true for KL), it contributes a term exponentially
larger than everything else combined, and one optimizer step later a different token plays
that role. The loss surface is a sequence of cliffs. And the §9 connection compounds it: §9
showed that *estimating* a divergence from samples goes wrong when the estimator involves
$r$ to the first power, because $\mathrm{Var}_q[r] = \chi^2$ can be enormous. An estimator of
$\chi^2$ itself involves $r^2$, whose variance is driven by the *fourth* moment of $r$. The
divergence whose value is "the variance of the thing that was already too noisy" is the last
thing you want as a training signal.

## Exercise 4: a k3-style estimator for the forward direction

**The exercise.** Derive the k3-style estimator for $\mathrm{KL}(p\|q)$ from samples of $q$,
using importance weighting (reweighting each sample's contribution by a probability ratio to
correct for drawing from $q$ rather than $p$). Measure its variance against the §9 setup and
explain why on-policy *forward*-KL distillation is harder than reverse.

**The approach.** First the derivation. The forward KL is an expectation under $p$, but the
samples come from $q$ (on-policy distillation samples from the student), so rewrite it with
the importance ratio $r(x) = p(x)/q(x)$:

$$\mathrm{KL}(p\|q) = \mathbb{E}_p[\log r] = \mathbb{E}_q[r \log r].$$

So the naive estimator averages $r \log r$ over draws from $q$. The k3 trick from §9 is to
subtract a *control variate*, a quantity with expectation zero that is correlated with the
noise: since $\mathbb{E}_q[r] = 1$, the term $(r - 1)$ averages to zero and can be subtracted
for free. That gives

$$k_3^{\mathrm{fwd}} = r \log r - (r - 1),$$

which is still unbiased, and is pointwise non-negative because
$t \log t - t + 1 \ge 0$ for all $t > 0$ (it is the forward-KL generator itself, shifted to
touch zero at $t = 1$; §5's table had $f(t) = t\log t$, and adding the affine term
$-(t-1)$ changes no divergence value but makes every sample non-negative). The measurement
plan mirrors §9 exactly: a close regime ($q \approx p$, late training) and a far regime
(cold start), comparing the naive and k3-style forward estimators, plus the *reverse*
direction's $k_1 = -\log r$ on the same draws as the yardstick for "how noisy is the
direction people actually run on-policy".

In [5]:
g = torch.Generator().manual_seed(7)
V, N = 500, 200_000

def run_forward_regime(tag, p, q):
    true_fwd = float((p * (p.log() - q.log())).sum())     # KL(p||q), full-vocab truth
    x = torch.multinomial(q, N, replacement=True, generator=g)
    log_r = (p.log() - q.log())[x]
    r = log_r.exp()
    ests = {"naive r*log r": r * log_r,
            "k3fwd r*log r - (r-1)": r * log_r - (r - 1),
            "reverse k1 = -log r": -log_r}                # estimates KL(q||p), for contrast
    print(f"{tag}: true KL(p||q) = {true_fwd:.5f} nats")
    print(f"{'estimator':>24} {'mean':>10} {'std':>12}")
    for name, k in ests.items():
        print(f"{name:>24} {k.mean():>10.5f} {k.std():>12.5f}")
    print()
    return true_fwd, ests

# Regime A: late training, student close to the teacher (same construction as lab 00 section 9).
z = torch.randn(V, generator=g) * 2.0
p_close = F.softmax(z, -1)
q_close = F.softmax(z + 0.3 * torch.randn(V, generator=g), -1)
kl_a, ea = run_forward_regime("A (q ~ p)", p_close, q_close)

# Regime B: cold start, student far from the teacher.
p_far = rand_dist(V, peak=2.0, generator=g)
q_far = rand_dist(V, peak=2.0, generator=g)
kl_b, eb = run_forward_regime("B (q far from p)", p_far, q_far)

for kl_t, ests in ((kl_a, ea), (kl_b, eb)):
    k3f = ests["k3fwd r*log r - (r-1)"]
    se = float(k3f.std()) / math.sqrt(N)
    assert (k3f >= -1e-6).all(), "every k3fwd sample is non-negative"
    assert abs(float(k3f.mean()) - kl_t) < 6 * se, "k3fwd is unbiased (within 6 standard errors)"
naive_a, naive_b = ea["naive r*log r"], eb["naive r*log r"]
k3f_a, k3f_b = ea["k3fwd r*log r - (r-1)"], eb["k3fwd r*log r - (r-1)"]
assert float(k3f_a.std()) < 0.25 * float(naive_a.std()), "close regime: control variate wins big"
assert float(k3f_b.std()) > 0.5 * float(naive_b.std()), "far regime: the r-tail dominates; no rescue"
assert float(naive_b.std()) > 20 * kl_b, "far regime: noise dwarfs the quantity being estimated"
assert float(naive_b.std()) > 10 * float(eb["reverse k1 = -log r"].std()), \
    "forward estimation is far noisier than reverse on the same draws"
print("checked: k3fwd unbiased and non-negative; variance verdict is regime-dependent")

A (q ~ p): true KL(p||q) = 0.05147 nats
               estimator       mean          std


           naive r*log r    0.05235      0.33631
   k3fwd r*log r - (r-1)    0.05145      0.05893


     reverse k1 = -log r    0.05233      0.33106



B (q far from p): true KL(p||q) = 4.74743 nats
               estimator       mean          std
           naive r*log r    4.06521    174.06255


   k3fwd r*log r - (r-1)    4.16531    150.07742
     reverse k1 = -log r    4.06940      2.21122



checked: k3fwd unbiased and non-negative; variance verdict is regime-dependent


**Interpretation.** Read the two printed blocks against the assertions.

In regime A (student close to teacher), the control variate is a clean win: the k3-style
estimator's standard deviation is several times below the naive $r\log r$ (the assertion
demands at least 4x, and the printed gap is wider), both means sit on the true KL, and every
single k3 sample is non-negative, so the running average never dips below zero the way a
noisy estimator's does. This mirrors §9's reverse-direction result: near agreement,
$r \approx 1$, the quadratic behaviour of $r\log r - (r-1)$ near $r=1$ keeps samples tiny,
and the estimator is excellent.

In regime B (cold start), both forward estimators are a disaster and the control variate
barely helps: the printed standard deviations are tens of times the true KL itself (the
assertion checks a factor of 20), meaning a batch-sized average of these samples is mostly
noise, and subtracting $(r-1)$ cannot rescue it because the noise *is* the heavy tail of $r$,
which appears in both terms. Note also the means in regime B visibly undershoot the truth:
the estimator is unbiased in expectation, but the expectation is carried by astronomically
rare draws with huge $r$, so any finite sample tends to sit low. An unbiased estimator you
cannot afford enough samples for behaves, in practice, like a biased one.

Why forward is harder than reverse, in one sentence the yardstick row makes concrete: the
reverse estimators need $r$ only inside a $\log$ (with the linear $(r-1)$ as an optional
correction), while the forward estimator needs $r$ *multiplicatively*, as the importance
weight itself, so the forward estimator's variance is governed by
$\mathbb{E}_q[r^2 \log^2 r]$, which Exercise 3 just showed is driven by the same
second-moment-of-$r$ quantity ($\chi^2$) that explodes whenever the models disagree. The
final assertion measures the gap: on identical draws in the far regime, the naive forward
estimator is more than 10x noisier than reverse $k_1$. That is Unit 07's `beta` discussion
arriving early: on-policy pipelines default to reverse-flavoured objectives not because
reverse is philosophically better but because it is the direction whose sampled estimator
does not require importance weights.